In [1]:
import pandas as pd
import numpy as np
import joblib
import shap

ROOT = r".."

FEATURES = [
    "followers",
    "following",
    "posts",
    "bio_length",
    "has_profile_picture",
    "is_private",
    "username_digit_ratio",
    "follower_following_ratio",
    "profile_completeness"
]

print("M20 - Interactive Explanation")
print("Initialized successfully!")

M20 - Interactive Explanation
Initialized successfully!


In [2]:
model = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_ensemble_v3.pkl"
)

df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

X = df[FEATURES]

print("Model loaded!")
print("Dataset shape:", df.shape)

Model loaded!
Dataset shape: (696, 10)


In [3]:
explainer = shap.TreeExplainer(model)

print("SHAP explainer ready!")

SHAP explainer ready!


In [4]:
def get_shap_explanations(profile):

    profile_df = pd.DataFrame(
        [[profile[feature] for feature in FEATURES]],
        columns=FEATURES
    )

    shap_values = explainer.shap_values(profile_df)

    # SHAP 0.52.0 format: (1, features, classes)
    fake_class_shap = shap_values[0, :, 1]

    explanation_df = pd.DataFrame({
        "feature": FEATURES,
        "value": profile_df.iloc[0].values,
        "shap_value": fake_class_shap
    })

    explanation_df["abs_shap"] = (
        explanation_df["shap_value"].abs()
    )

    explanation_df = explanation_df.sort_values(
        "abs_shap",
        ascending=False
    ).reset_index(drop=True)

    return explanation_df

In [5]:
def get_top_shap_factors(profile, top_n=3):

    explanation_df = get_shap_explanations(profile)

    top_factors = []

    for _, row in explanation_df.head(top_n).iterrows():

        if row["shap_value"] > 0:
            impact = "increases risk"
        elif row["shap_value"] < 0:
            impact = "decreases risk"
        else:
            impact = "has little effect"

        top_factors.append({
            "feature": row["feature"],
            "value": float(row["value"]),
            "shap_value": round(
                float(row["shap_value"]), 4
            ),
            "impact": impact
        })

    return top_factors

In [6]:
def get_nlp_explanation(nlp_result):

    if not nlp_result:
        return {
            "nlp_risk": 0.0,
            "matched_categories": [],
            "matched_phrases": []
        }

    matched_phrases = []

    for match in nlp_result.get("matches", []):

        matched_phrases.append({
            "category": match["category"],
            "matched_text": match["matched_text"]
        })

    return {
        "nlp_risk": nlp_result.get("score", 0.0),
        "matched_categories": list(
            set(
                item["category"]
                for item in matched_phrases
            )
        ),
        "matched_phrases": matched_phrases
    }

In [11]:
import pandas as pd
import joblib
import shap

model = joblib.load(
    ROOT + r"\backend\trained_models\authentiscore_ensemble_v3.pkl"
)

df = pd.read_csv(
    ROOT + r"\data\processed\instagram_features.csv"
)

X = df[FEATURES]

explainer = shap.TreeExplainer(model)

print("Model loaded!")
print("Dataset shape:", df.shape)
print("SHAP explainer ready!")

Model loaded!
Dataset shape: (696, 10)
SHAP explainer ready!


In [12]:
def calculate_ml_risk(profile):

    X_profile = pd.DataFrame(
        [[profile[feature] for feature in FEATURES]],
        columns=FEATURES
    )

    probability = model.predict_proba(X_profile)[0][1]

    return round(float(probability * 100), 2)

In [13]:
def calculate_profile_rule_risk(profile):

    risk_score = 0
    signals = []

    if profile["following"] > 0:

        ratio = profile["followers"] / profile["following"]

        if ratio < 0.1:
            risk_score += 20

            signals.append({
                "rule": "low_follower_following_ratio",
                "weight": 20,
                "message": "Very low follower-to-following ratio"
            })

    if profile["username_digit_ratio"] > 0.5:

        risk_score += 15

        signals.append({
            "rule": "high_username_digit_ratio",
            "weight": 15,
            "message": "Username contains a high proportion of digits"
        })

    if profile["has_profile_picture"] == 0:

        risk_score += 20

        signals.append({
            "rule": "missing_profile_picture",
            "weight": 20,
            "message": "Profile picture is missing"
        })

    if profile["profile_completeness"] < 0.6:

        risk_score += 15

        signals.append({
            "rule": "low_profile_completeness",
            "weight": 15,
            "message": "Profile has low completeness"
        })

    if profile["posts"] == 0:

        risk_score += 10

        signals.append({
            "rule": "no_posts",
            "weight": 10,
            "message": "Profile has no posts"
        })

    if profile["bio_length"] == 0:

        risk_score += 10

        signals.append({
            "rule": "empty_bio",
            "weight": 10,
            "message": "Profile has no biography"
        })

    return {
        "profile_rule_risk": min(risk_score, 100),
        "signals": signals
    }

In [14]:
def calculate_unified_risk(
    ml_risk,
    nlp_risk,
    rule_risk
):

    final_score = (
        ml_risk * 0.60
        + nlp_risk * 0.25
        + rule_risk * 0.15
    )

    return round(
        min(max(final_score, 0), 100),
        2
    )

In [15]:
def classify_risk(score):

    if score < 30:
        return "Low Risk"

    elif score < 60:
        return "Medium Risk"

    elif score < 80:
        return "High Risk"

    else:
        return "Very High Risk"

In [16]:
def get_top_shap_factors(profile, top_n=3):

    profile_df = pd.DataFrame(
        [[profile[feature] for feature in FEATURES]],
        columns=FEATURES
    )

    shap_values = explainer.shap_values(profile_df)

    # SHAP 0.52.0:
    # shape = (1, features, classes)
    fake_class_shap = shap_values[0, :, 1]

    explanation_df = pd.DataFrame({
        "feature": FEATURES,
        "value": profile_df.iloc[0].values,
        "shap_value": fake_class_shap
    })

    explanation_df["abs_shap"] = (
        explanation_df["shap_value"].abs()
    )

    explanation_df = explanation_df.sort_values(
        "abs_shap",
        ascending=False
    )

    top_factors = []

    for _, row in explanation_df.head(top_n).iterrows():

        if row["shap_value"] > 0:
            impact = "increases risk"
        elif row["shap_value"] < 0:
            impact = "decreases risk"
        else:
            impact = "has little effect"

        top_factors.append({
            "feature": row["feature"],
            "value": float(row["value"]),
            "shap_value": round(
                float(row["shap_value"]), 4
            ),
            "impact": impact
        })

    return top_factors

In [17]:
def generate_explanation(
    profile,
    final_score,
    classification,
    ml_risk,
    nlp_risk,
    rule_result,
    nlp_result=None
):

    top_factors = get_top_shap_factors(
        profile,
        top_n=3
    )

    nlp_signals = {
        "nlp_risk": float(nlp_risk),
        "matched_categories": [],
        "matched_phrases": []
    }

    if nlp_result is not None:

        matches = nlp_result.get("matches", [])

        nlp_signals["matched_phrases"] = matches

        nlp_signals["matched_categories"] = list(
            set(
                match["category"]
                for match in matches
            )
        )

    return {
        "final_score": round(float(final_score), 2),

        "classification": classification,

        "risk_components": {
            "ml_risk": round(float(ml_risk), 2),
            "nlp_risk": round(float(nlp_risk), 2),
            "rule_risk": round(
                float(rule_result["profile_rule_risk"]),
                2
            )
        },

        "top_ml_factors": top_factors,

        "nlp_signals": nlp_signals,

        "rule_signals": rule_result["signals"]
    }

In [18]:

profile = df.iloc[0]

ml_risk = calculate_ml_risk(profile)

rule_result = calculate_profile_rule_risk(profile)

rule_risk = rule_result["profile_rule_risk"]

nlp_risk = 0

final_score = calculate_unified_risk(
    ml_risk,
    nlp_risk,
    rule_risk
)

classification = classify_risk(final_score)

result = generate_explanation(
    profile=profile,
    final_score=final_score,
    classification=classification,
    ml_risk=ml_risk,
    nlp_risk=nlp_risk,
    rule_result=rule_result
)

result


{'final_score': 2.2,
 'classification': 'Low Risk',
 'risk_components': {'ml_risk': 3.67, 'nlp_risk': 0.0, 'rule_risk': 0.0},
 'top_ml_factors': [{'feature': 'posts',
   'value': 32.0,
   'shap_value': -0.1604,
   'impact': 'decreases risk'},
  {'feature': 'followers',
   'value': 1000.0,
   'shap_value': -0.1425,
   'impact': 'decreases risk'},
  {'feature': 'follower_following_ratio',
   'value': 1.0471204188481675,
   'shap_value': -0.0917,
   'impact': 'decreases risk'}],
 'nlp_signals': {'nlp_risk': 0.0,
  'matched_categories': [],
  'matched_phrases': []},
 'rule_signals': []}

In [19]:
import json

print(json.dumps(result, indent=4))

{
    "final_score": 2.2,
    "classification": "Low Risk",
    "risk_components": {
        "ml_risk": 3.67,
        "nlp_risk": 0.0,
        "rule_risk": 0.0
    },
    "top_ml_factors": [
        {
            "feature": "posts",
            "value": 32.0,
            "shap_value": -0.1604,
            "impact": "decreases risk"
        },
        {
            "feature": "followers",
            "value": 1000.0,
            "shap_value": -0.1425,
            "impact": "decreases risk"
        },
        {
            "feature": "follower_following_ratio",
            "value": 1.0471204188481675,
            "shap_value": -0.0917,
            "impact": "decreases risk"
        }
    ],
    "nlp_signals": {
        "nlp_risk": 0.0,
        "matched_categories": [],
        "matched_phrases": []
    },
    "rule_signals": []
}
